# 02 — Model: sklearn HistGradientBoostingClassifier

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.
No extra installs needed — this one runs with plain scikit-learn.

Saves `oof_hgb.csv` and `test_pred_hgb.csv` for the ensembling notebook.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

from sklearn.ensemble import HistGradientBoostingClassifier
import time

cat_idx = [X.columns.get_loc(c) for c in cat_cols]


In [ ]:
oof_hgb = np.zeros(len(X))
test_hgb = np.zeros(len(Xtest))

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = HistGradientBoostingClassifier(
        categorical_features=cat_idx, learning_rate=0.05, max_leaf_nodes=255,
        max_iter=500, l2_regularization=2.0, min_samples_leaf=50,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=25, random_state=SEED
    )
    model.fit(X.iloc[tr_idx], y[tr_idx])
    p_va = model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_hgb[va_idx] = p_va
    test_hgb += model.predict_proba(Xtest)[:, 1] / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  ({time.time()-t0:.0f}s elapsed)")

print("HGB OOF AUC:", roc_auc_score(y, oof_hgb))


In [ ]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_hgb}).to_csv(f"{DATA_DIR}/oof_hgb.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_hgb}).to_csv(f"{DATA_DIR}/test_pred_hgb.csv", index=False)
print("saved oof_hgb.csv and test_pred_hgb.csv")
